# 07. Agent 기본 실습

- Agent는 LLM이 스스로 Tool을 선택하고 실행하며 목표를 달성함
- LangGraph의 `create_react_agent`를 사용함
- **ReAct 패턴**: Reasoning(추론) → Acting(행동) → Observation(관찰) 반복

In [4]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)
if not os.getenv("LLM_API_KEY"):
    raise ValueError("프로젝트 .env에 LLM_API_KEY를 설정하세요.")


## 1. Tool 준비

In [5]:
from langchain_core.tools import tool
from datetime import datetime

@tool
def add(a: int, b: int) -> int:
    """두 정수를 더합니다."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """두 정수를 곱합니다."""
    return a * b

@tool
def get_current_time() -> str:
    """현재 날짜와 시간을 반환합니다."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@tool
def get_weather(city: str) -> str:
    """도시 이름을 받아 날씨 정보를 반환합니다."""
    weather_data = {"서울": "맑음, 22도", "부산": "흐림, 18도", "제주": "비, 16도"}
    return weather_data.get(city, f"{city}의 날씨 정보를 찾을 수 없습니다.")

tools = [add, multiply, get_current_time, get_weather]

## 2. Agent 생성
- create_agent() 함수 : Tool 사용이 가능한 Agent 그래프를 빠르게 만들어주는 함수
- create_react_agent() 이후 버전에 삭제 될 예정임

In [6]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

# 1. LLM 생성
llm = ChatOpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.getenv("LLM_BASE_URL", "https://monogpt.kr/api/monorouter/v1"), use_responses_api=False, 
    model="gpt-4.1",
    temperature=0
)

# 2. Agent 생성
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="당신은 유용한 AI 어시스턴트입니다. 필요한 경우 Tool을 사용해 정확한 정보를 제공하세요."
)

print("Agent 생성 완료")

Agent 생성 완료


- 시퀀스 다이어그램
```mermaid
sequenceDiagram
    participant User as 사용자
    participant Agent as AI Agent
    participant LLM as LLM
    participant Tool as Tool

    User->>Agent: 질문 입력
    Agent->>LLM: 질문 + Tool 목록 전달

    LLM->>LLM: 질문 분석
    LLM-->>Agent: Tool 호출 요청 반환

    Agent->>Tool: Tool 실행
    Tool-->>Agent: Tool 결과 반환

    Agent->>LLM: Tool 결과 전달
    LLM-->>Agent: 최종 답변 생성

    Agent-->>User: 최종 답변 반환
```

## 3. Agent 실행 - 단순 질문

In [7]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "지금 몇시야?"
        }
    ]
})

print(result["messages"][-1].content)

지금은 2026년 10월 1일 오후 2시 8분입니다.


## 4. Agent 실행 - 계산이 필요한 질문

In [8]:
result = agent.invoke({"messages": [("human", "25 곱하기 4는 얼마야?")]})

print(result["messages"][-1].content)

25 곱하기 4는 100입니다.


## 5. Agent 실행 - 복합 질문 (여러 Tool 사용)

In [9]:
result = agent.invoke({
    "messages": [("human", "서울 날씨를 알려주고, 15 더하기 27도 계산해줘")]
})

print(result["messages"][-1].content)

서울의 현재 날씨는 맑고, 기온은 22도입니다.
또한, 15 더하기 27의 계산 결과는 42입니다.


## 6. Agent 실행 과정 확인

중간 단계(추론, Tool 호출, 결과)를 모두 출력합니다.

In [10]:
result = agent.invoke({"messages": [("human", "부산 날씨는 어때?")]})

for msg in result["messages"]:
    role = msg.__class__.__name__
    print(f"[{role}]")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print("  Tool 호출:", msg.tool_calls)
    else:
        print(" ", msg.content)
    print()

[HumanMessage]
  부산 날씨는 어때?

[AIMessage]
  Tool 호출: [{'name': 'get_weather', 'args': {'city': '부산'}, 'id': 'call_79VaXqMyetZQuMPqUKJLJdjq', 'type': 'tool_call'}]

[ToolMessage]
  흐림, 18도

[AIMessage]
  부산의 현재 날씨는 흐리고, 기온은 18도입니다. 외출 시 참고하세요!



## 7. 스트리밍으로 Agent 실행

In [11]:
for chunk in agent.stream({"messages": [("human", "지금 시간과 서울 날씨 알려줘")]}):
    # print(f"새로운 응답 청크 도착: {chunk}")
    for key, value in chunk.items():
        if "messages" in value:
            last_msg = value["messages"][-1]
            print(f"[{key}] {last_msg.content or last_msg.tool_calls}")

[model] [{'name': 'get_current_time', 'args': {}, 'id': 'call_SGhnUGCRQE9ZixIeFOabz7RX', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'city': '서울'}, 'id': 'call_InWgsAVzSrzm1wehXL2FA6wH', 'type': 'tool_call'}]
[tools] 2026-10-01 14:08:52
[tools] 맑음, 22도
[model] 현재 시간은 2026년 10월 1일 14시 8분 52초이고, 서울의 날씨는 맑고 기온은 22도입니다.
